# Correcting Fixation Drift



The pymovements library provides a collection of built-in fixation drift correction algorithms to be used on gaze data.

These algorithms use various methods to automatically correct fixations that are deemed inaccurate due to measurement issues such as noise, slope, or shift.

In this tutorial you'll learn how to:

- load a reading dataset and prepare fixations
- map fixations to the AOIs
- apply fixation drift correction algorithms, along with the WoC ensemble
- pick a specific algorithm for drift correction
- visualize the results before and after correction

All examples use the GGTG Dataset that comes with pymovements

## Loading Data

In [ ]:
from pathlib import Path

import polars as pl

import pymovements
from pymovements.events.correction.fixation_correction import \
    correct_fixation_locations

dataset = pymovements.Dataset("GGTG", path="data/GGTG")

dataset.download()

dataset.load(
    subset={"subject_id": "P01"},
)

## Extracting fixations from data 

Our dataset already has text stimuli, which contain the words and their locations, of the stimulus the fixations were measured on, but we first need to extract the fixations so we can then map them to their respective text stimuli. 

In [ ]:
stimulus_name = "goldfish-pos.text.0"

fixation_path = (
    dataset.paths.precomputed_events
    / dataset.fileinfo["precomputed_events"]["filepath"][0]
)

fixations = (
    pl.read_csv(fixation_path)
    .filter(pl.col("stimulus") == stimulus_name)
    .select(
        "stimulus",
        "onset",
        "offset",
        "duration",
        "location_x",
        "location_y",
    )
    .with_columns(pl.lit("fixation").alias("name"))
)

## Extracting text stimulus and creating a TextStimulus object

To provide a visualization for us to plot our fixations onto, we must first prepare a text_stimulus object.

In [ ]:
aoi_path = (
    dataset.paths.stimuli
    / dataset.fileinfo["textstimulus"]
    .filter(
        (pl.col("stimulus") == stimulus_name)
        & (pl.col("unit") == "word")
    )["filepath"][0]
)

aois = pl.read_csv(aoi_path)

text_stimulus = pymovements.stimulus.TextStimulus(
    aois=aois,
    aoi_column="content",
    start_x_column="left",
    start_y_column="top",
    end_x_column="right",
    end_y_column="bottom",
)

## Applying fixation correction algorithms

To apply our fixation correction algorithms, we first create Events objects from our extracted fixations, after which we can apply correct_fixations() to them in the following forms with various algorithms depending on the user's choice.

In [ ]:
events_raw = pymovements.Events(fixations)
events_woc = pymovements.Events(fixations)
events_warp = pymovements.Events(fixations)
events_chain = pymovements.Events(fixations)

events_woc.correct_fixations(
    text_stimulus,
    algorithm="wisdom_of_the_crowd",
)

events_warp.correct_fixations(
    text_stimulus,
    algorithm="warp",
)

events_chain.correct_fixations(
    text_stimulus,
    algorithm="chain",
)

## Plotting pre and post-correction scanpaths

Before plotting our scanpaths, we concatenate our location columns as scanpathplot() expects a single column of [x, y] coordinates. After this, we add the events to a Gaze object and pass it to scanpathplot() for our final plots.

In [ ]:
events_raw = pymovements.Events(
    events_raw.frame.with_columns(
        pl.concat_list(["location_x", "location_y"]).alias("location"),
    ),
)

gaze_for_plot = pymovements.Gaze(events=events_raw)

fig, ax = text_stimulus.plot()

fig, ax = pymovements.plotting.scanpathplot(
    gaze=gaze_for_plot,
    position_column="location",
    event_name="fixation",
    ax=ax,
    color="red",
    alpha=0.7,
    add_arrows=True,
    title="Raw fixation scanpath",
)

Plotting the raw scanpaths we can see how some fixations are slightly off words and either up or down, if we plot the fixation corrected scanpaths we can see the difference 

In [ ]:
events_woc = pymovements.Events(
    events_woc.frame.with_columns(
        pl.concat_list(["location_x", "location_y"]).alias("location"),
    ),
)

gaze_for_plot = pymovements.Gaze(events=events_woc)

fig, ax = text_stimulus.plot()

fig, ax = pymovements.plotting.scanpathplot(
    gaze=gaze_for_plot,
    position_column="location",
    event_name="fixation",
    ax=ax,
    color="red",
    alpha=0.7,
    add_arrows=True,
    title="Corrected fixation scanpath",
)